In [1]:
import numpy as np
import obspy
import emd
import pandas as pd
from tqdm.notebook import tqdm
import os
import scipy.signal as sg
from concurrent.futures import ThreadPoolExecutor
%matplotlib inline
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import clear_output
from torchinfo import summary

In [2]:
#Comprobamos que CUDA esté corriendo correctamente en nuestra pc
print("CUDA Available: ", torch.cuda.is_available())
print("CUDA Version: ", torch.version.cuda)
print("Device Count: ", torch.cuda.device_count())

CUDA Available:  True
CUDA Version:  12.4
Device Count:  1


In [ ]:
# Función de filtro pasa-banda utilizando el diseño de filtro Butterworth.
# Filtra los datos entre las frecuencias lowc y high.
def butter_bandpass_filter(senal: np.array, lowcut: float, highcut: float, fs: float, order: int):
    nyquist = 0.5 * fs  # Frecuencia de Nyquist, la mitad de la tasa de muestreo
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = sg.butter(order, [low, high], btype='band', analog=False)  # Diseña el filtro pasa-banda Butterworth
    y = sg.filtfilt(b, a, senal)  # Aplica el filtro a los datos usando filtrado cero-fase
    return y

In [ ]:
def aplicar_filtro_notch(senal, fs, f0, Q):
    """
    senal : array-like
        La señal de entrada a la que se le aplicará el filtro.
    fs : float
        Frecuencia de muestreo de la señal (en Hz).
    f0 : float
        Frecuencia central que se desea eliminar (en Hz).
    Q : float
        Factor de calidad del filtro notch.
    Retorna:
    --------
    senal_filtrada : array-like
        La señal después de aplicar el filtro notch.
    """
    # Diseñar el filtro notch
    b, a = signal.iirnotch(f0, Q, fs)

    # Aplicar el filtro a la señal
    senal_filtrada = signal.lfilter(b, a, senal)

    return senal_filtrada

In [ ]:
# Ruta de la carpeta donde están los archivos de audio
folder_path = "./Respiratory_Sound_Database\Respiratory_Sound_Database//audio_and_txt_files"
# Obtener lista de archivos de audio en la carpeta
audio_files = [f for f in os.listdir(folder_path) if f.endswith(".wav")]

audio_path = os.path.join(folder_path, audio_files[7])

In [ ]:

audio, sr = librosa.load(audio_path, sr=None)  # sr=None para mantener la tasa de muestreo original
time = np.arange(len(audio))/sr